In [44]:
import pandas as pd

In [45]:
input_paths = [
    "../Data/0_example.txt",
    "../Data/1_binary_landscapes.txt",
    "../Data/10_computable_moments.txt",
    "../Data/11_randomizing_paintings.txt",
    "../Data/110_oily_portraits.txt"
    ]

output_paths = [
        "../output/0_example.txt",
        "../output/1_binary_landscapes.txt",
        "../output/10_computable_moments.txt",
        "../output/11_randomizing_paintings.txt",
        "../output/110_oily_portraits.txt"
    ]


In [46]:
file_number = 0

input_path = input_paths[file_number]
output_path = output_paths[file_number]

with open(input_path, 'r') as file:
    lines = file.read().strip().split('\n')
lines = lines[1:]

In [47]:

parsed = []
index = 0
for line in lines:
    parts = line.strip().split()
    painting_type = parts[0]
    num_tags = int(parts[1])
    tags = set(parts[2:]) 
    parsed.append({
        "index" : index,
        "Type": painting_type,
        "Num_Tags": num_tags,
        "Tags": tags
    })
    index += 1
df = pd.DataFrame(parsed)
# df = df.sort_values(by='Type', ascending=False)
print(df)

   index Type  Num_Tags                     Tags
0      0    L         3     {fear, animals, war}
1      1    P         2           {smile, woman}
2      2    P         2           {woman, pearl}
3      3    L         3  {fear, survivors, raft}


In [39]:

df_p = df[df['Type'] == 'P'].copy()
df_l = df[df['Type'] == 'L'].copy()
merged = []
for i in range(0, len(df_p), 2):
    if i + 1 < len(df_p):
        idx1, idx2 = df_p.iloc[i]['index'], df_p.iloc[i+1]['index']
        tags1, tags2 = df_p.iloc[i]['Tags'], df_p.iloc[i+1]['Tags']
        tags3 = tags1.union(tags2)
        merged.append({
                "index": f"{idx1} {idx2}",
                "Type" : "P",
                "Tags": tags3,
                "Num_Tags" : len(tags3)
            })


In [ ]:
merged_df = pd.DataFrame(merged)
df = pd.concat([merged_df, df_l], ignore_index=True)
print(df)

# 1
# 3
# 2
# 4
# 6

#17 sec 1000 rows * 80 = 22 minutes
#24 hours


       index Type                                               Tags  Num_Tags
0        1 3    P  {w52, w82, ww6, wv3, wc1, wz2, wx4, wr5, w47, ...        25
1        5 6    P  {w52, wd, wv4, w02, ws2, w22, wz, wn6, wq1, wn...        22
2        7 8    P  {w52, wp, wl, w6, wx, w37, w53, wq6, wb7, wp2,...        16
3      10 12    P  {wf1, w02, wx2, wr5, wb5, wc5, ww5, wc4, wm, w...        23
4      13 14    P  {w52, w82, w62, wd2, wz2, wx4, wr5, ww5, w01, ...        20
...      ...  ...                                                ...       ...
59995  89988    L  {wm2, wf1, w85, wk5, wr5, wp1, wb2, wb1, ww3, ...        11
59996  89990    L  {w52, wq1, w82, ww6, w85, ws2, wh6, wb1, w01, ...        10
59997  89992    L  {w34, wm2, wn3, w05, wq6, wp6, wq3, wp1, wc5, ...        13
59998  89996    L                               {wp6, wf1, w34, w94}         4
59999  89997    L  {wf1, wq2, w6, wr6, wq6, wh6, wr5, w87, wc5, w...        12

[60000 rows x 4 columns]


In [41]:
# Greedy algorithm
def greedy_reorder(df):
    used = set()
    order = []
    
    # Start with the row with max total tag overlap with others
    start = max(df.index, key=lambda i: sum(len(df.loc[i, 'Tags'] & df.loc[j, 'Tags']) for j in df.index if i != j))
    current = start
    used.add(current)
    order.append(current)

    while len(used) < len(df):
        next_index = max(
            (i for i in df.index if i not in used),
            key=lambda i: len(df.loc[current, 'Tags'] & df.loc[i, 'Tags']),
            default=None
        )
        if next_index is None:
            break
        used.add(next_index)
        order.append(next_index)
        current = next_index
    
    return df.loc[order].reset_index(drop=True)

#
chunk_size = 1000
chunks = [df[i:i + chunk_size].reset_index(drop=True) for i in range(0, len(df), chunk_size)]

processed_chunks = []
for chunk in chunks:
    chunk = greedy_reorder(chunk)
    processed_chunks.append(chunk)

final_df = pd.concat(processed_chunks, ignore_index=True)



In [42]:
import winsound    

winsound.Beep(1440, 200)  

In [43]:
output = final_df['index']  
len_output = len(output)

with open(output_path, "w") as f:
    f.write(str(len_output) + "\n")
    for line in output.values:
        f.write(str(line)+'\n')

212843